In [25]:

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import heapq
import math
import time
import ipywidgets as widgets
from IPython.display import display, clear_output
import folium

In [27]:

df = pd.read_csv('/content/drive/MyDrive/Artificial_Intelligence/DataSet/indian-cities-dataset.csv')

df.head()


,Origin,Destination,Distance
0,Agra,Delhi,240
1,Agra,Lucknow,334
2,Agra,Kanpur,277
3,Ahmedabad,Mumbai,526
4,Ahmedabad,Pune,663


In [28]:
graph = nx.Graph()

for _, row in df.iterrows():
    graph.add_edge(row['Origin'], row['Destination'], weight=row['Distance'])

print("Total Cities:", len(graph.nodes()))

Total Cities: 20


In [29]:
city_coordinates = {
    'Mumbai': (19.0760, 72.8777), 'Delhi': (28.7041, 77.1025),
    'Bengaluru': (12.9716, 77.5946), 'Chennai': (13.0827, 80.2707),
    'Kolkata': (22.5726, 88.3639), 'Hyderabad': (17.3851, 78.4867),
    'Ahmedabad': (23.0225, 72.5714), 'Pune': (18.5204, 73.8567),
    'Surat': (21.1702, 72.8311), 'Jaipur': (26.9124, 75.7873),
    'Lucknow': (26.8467, 80.9462), 'Kanpur': (26.4650, 80.3498),
    'Nagpur': (21.1452, 79.0882), 'Indore': (22.7196, 75.8577),
    'Bhopal': (23.2599, 77.4126), 'Patna': (25.5941, 85.1376),
    'Ludhiana': (30.9010, 75.8573), 'Agra': (27.1767, 78.0081),
    'Vadodara': (22.3072, 73.1812), 'Nashik': (19.9975, 73.7898),
    'Goa': (15.4909, 73.8278), 'Udaipur': (24.579,73.712),
    'Kochi': (9.9312, 76.2673), 'Thiruvananthapuram': (8.5241, 76.9366),
    'Vishakhapatnam': (17.7197,83.3119), 'Bhubaneswar': (20.2961, 85.8245),
    'Varanasi': (25.3176, 82.9739)
}

In [30]:
def heuristic(c1, c2):
    lat1, lon1 = city_coordinates[c1]
    lat2, lon2 = city_coordinates[c2]
    R = 6371

    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)

    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1-a))

In [31]:
def astar(graph, start, end):
    open_list = []
    heapq.heappush(open_list, (0, 0, start, [start]))
    visited = set()
    expanded_nodes = []

    while open_list:
        f, g, node, path = heapq.heappop(open_list)

        expanded_nodes.append(node)

        if node == end:
            return path, g, len(visited), expanded_nodes

        if node in visited:
            continue

        visited.add(node)

        for nbr in graph.neighbors(node):
            w = graph[node][nbr]['weight']
            new_g = g + w
            new_f = new_g + heuristic(nbr, end)

            heapq.heappush(open_list, (new_f, new_g, nbr, path+[nbr]))

    return None, float('inf'), 0, []

In [32]:
def dijkstra(graph, start, end):
    path = nx.dijkstra_path(graph, start, end)
    dist = nx.dijkstra_path_length(graph, start, end)
    return path, dist

In [33]:
def draw_map(path, start, end):
    center = city_coordinates[start]
    m = folium.Map(location=center, zoom_start=5)

    coords = [city_coordinates[c] for c in path]

    folium.PolyLine(coords, color="blue", weight=5).add_to(m)

    for city in path:
        folium.Marker(
            location=city_coordinates[city],
            popup=city
        ).add_to(m)

    return m

In [34]:
def animate_path(path):
    plt.figure(figsize=(8,5))
    pos = {c:(lon,lat) for c,(lat,lon) in city_coordinates.items()}

    for i in range(1, len(path)+1):
        plt.clf()

        nx.draw(graph, pos, node_size=200, alpha=0.2)

        partial = path[:i]
        edges = list(zip(partial[:-1], partial[1:]))

        nx.draw_networkx_edges(graph, pos, edgelist=edges, edge_color='red', width=3)

        plt.title(f"Step {i}: {partial}")
        plt.pause(0.8)

    plt.show()

In [35]:
def run_app():

    cities = sorted(graph.nodes())
    cities_with_default = ["Select City"] + cities

    start_dd = widgets.Dropdown(options=cities_with_default, description="Start:")
    end_dd = widgets.Dropdown(options=cities_with_default, description="End:")
    btn = widgets.Button(description="Find Path", button_style='success')
    out = widgets.Output()

    def on_click(b):
        with out:
            clear_output()

            s = start_dd.value
            e = end_dd.value

            if s == "Select City" or e == "Select City":
                print("⚠️ Select both cities")
                return

            if s == e:
                print("⚠️ Same city")
                return

            # A*
            t1 = time.time()
            p1, d1, v1, expanded = astar(graph, s, e)
            t2 = time.time()

            # Dijkstra
            t3 = time.time()
            p2, d2 = dijkstra(graph, s, e)
            t4 = time.time()

            # 📊 METRICS TABLE (IMPROVED)
            print("===== PERFORMANCE =====")
            display(pd.DataFrame({
                "Algorithm": ["A*", "Dijkstra"],
                "Distance": [d1, d2],
                "Time": [round(t2-t1,4), round(t4-t3,4)],
                "Visited Nodes": [v1, "-"]
            }))

            print("\n===== EXPANDED NODES (A*) =====")
            print(expanded[:15], "...")

            # 🟦 MATPLOTLIB GRAPH (same style)
            plt.figure(figsize=(12,7))
            pos = {c:(lon,lat) for c,(lat,lon) in city_coordinates.items()}

            nx.draw(graph, pos, node_size=400, alpha=0.2)

            if p1:
                edges1 = list(zip(p1[:-1], p1[1:]))
                nx.draw_networkx_edges(graph, pos, edgelist=edges1, edge_color='red', width=3)

            if p2:
                edges2 = list(zip(p2[:-1], p2[1:]))
                nx.draw_networkx_edges(graph, pos, edgelist=edges2, edge_color='blue', style='dashed')

            nx.draw_networkx_nodes(graph, pos, nodelist=[s], node_color='green', node_size=600)
            nx.draw_networkx_nodes(graph, pos, nodelist=[e], node_color='orange', node_size=600)

            plt.title(f"{s} → {e}")
            plt.axis('off')
            plt.show()

            # 🌍 FOLIUM MAP (BIG IMPACT)
            print("===== REAL MAP VIEW =====")
            display(draw_map(p1, s, e))

            # 🎞️ ANIMATION (optional)
            print("===== ANIMATION =====")
            animate_path(p1)

    btn.on_click(on_click)

    display(start_dd, end_dd, btn, out)

run_app()

Dropdown(description='Start:', options=('Select City', 'Agra', 'Ahmedabad', 'Bengaluru', 'Bhubaneswar', 'Chenn…

Dropdown(description='End:', options=('Select City', 'Agra', 'Ahmedabad', 'Bengaluru', 'Bhubaneswar', 'Chennai…

Button(button_style='success', description='Find Path', style=ButtonStyle())

Output()